In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/Bengaluru_House_Data - Bengaluru_House_Data.csv")


In [15]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00,4.0
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,Unknown,1440,2.0,3.0,62.00,3.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00,3.0
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,Unknown,1200,2.0,1.0,51.00,2.0


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
)

In [4]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [17]:
df["location"] = df["location"].fillna("Unknown")
df["size"] = df["size"].fillna("Unknown")
df["society"] = df["society"].fillna("Unknown")

df["bath"] = df["bath"].fillna(df["bath"].median())
df["balcony"] = df["balcony"].fillna(df["balcony"].median())

In [16]:
df = df.drop_duplicates()


In [18]:
df.isnull().sum()

,0
area_type,0
availability,0
location,0
size,0
society,0
total_sqft,0
bath,0
balcony,0
price,0
bhk,16


In [6]:
# No additional dataset is available, so merging is not required.

In [7]:
df["bhk"] = df["size"].str.extract(r"(\d+)").astype(float)

three_bhk = df[df["bhk"] >= 3]

three_bhk.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.0,4.0
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,Unknown,1440,2.0,3.0,62.0,3.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.0,3.0
6,Super built-up Area,18-May,Old Airport Road,4 BHK,Jaades,2732,4.0,2.0,204.0,4.0
7,Super built-up Area,Ready To Move,Rajaji Nagar,4 BHK,Brway G,3300,4.0,2.0,600.0,4.0


In [8]:
encoded_df = pd.get_dummies(
    df,
    columns=["area_type"],
    drop_first=True
)

encoded_df.head()

,availability,location,size,society,total_sqft,bath,balcony,price,bhk,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
0,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07,2.0,False,False,True
1,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00,4.0,False,True,False
2,Ready To Move,Uttarahalli,3 BHK,Unknown,1440,2.0,3.0,62.00,3.0,False,False,False
3,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00,3.0,False,False,True
4,Ready To Move,Kothanur,2 BHK,Unknown,1200,2.0,1.0,51.00,2.0,False,False,True


In [9]:
average_price_area = df.groupby("area_type")["price"].mean()

average_price_area

,price
area_type,
Built-up Area,104.822222
Carpet Area,89.502356
Plot Area,210.520543
Super built-up Area,94.319615


In [20]:
location_summary = (
    df.groupby("location")
    .agg(
        average_price=("price", "mean"),
        number_of_properties=("price", "count")
    )
    .reset_index()
)

location_summary.head(10)

,location,average_price,number_of_properties
0,1 Annasandrapalya,150.0,1
1,1 Giri Nagar,227.5,1
2,1 Immadihalli,150.0,1
3,1 Ramamurthy Nagar,170.0,1
4,12th cross srinivas nagar banshankari 3rd stage,200.0,1
5,1A Block Koramangala,227.5,1
6,1Channasandra,125.0,1
7,1Hanuman Nagar,227.5,1
8,1Hoysalanagar,227.5,1
9,1Kasavanhalli,200.0,1


In [11]:
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[
    (df["price"] < lower_limit) |
    (df["price"] > upper_limit)
]

len(outliers)

1257

In [12]:
df["price"] = df["price"].clip(
    lower=lower_limit,
    upper=upper_limit)

In [14]:
print("\nFinal Dataset:")
df.head()


Final Dataset:


,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00,4.0
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,Unknown,1440,2.0,3.0,62.00,3.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00,3.0
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,Unknown,1200,2.0,1.0,51.00,2.0
